# 03 — Pembersihan, organisasi & kualitas metadata (Bab 3.4, 4.1)

**Tujuan:** dari 3 CSV mentah di raw zone → **1 tabel metadata bersih** (Parquet) di `/datalake/processed/metadata/`, dipartisi per `dataset`, plus **laporan kualitas data** sebelum vs sesudah.

Aturan cleaning:

| # | Aturan | Dataset |
|---|---|---|
| R1 | Label ComCat mengikuti **pemetaan resmi PNSN → ComCat** (Ni dkk. 2023, bagian *Event Type* + Tabel S2). Beda nama kode saja (mis. `lf`, `px`) **bukan** inkonsistensi | comcat |
| R2 | Konflik sungguhan earthquake ↔ explosion → **label PNSN dipakai** (saran README) | comcat |
| R3 | Label PNSN di luar earthquake/explosion (`su`, `sn`, `uk`, `qb`, …) → **dibuang**, bukan di-relabel (window ComCat 150 s selaras origin time ≠ Exotic 180 s selaras P) | comcat |
| R4 | Kelas di luar 4 target (sonic boom, thunder, plane crash) → dibuang (tetap ada di raw) | exotic |
| R5 | `trace_name` duplikat → **semua baris dalam pasangan dibuang** (tidak bisa tahu mana yang cocok dengan waveform) | noise |
| R6 | Validasi: sampling rate = 100 Hz, jumlah sampel sesuai dataset, pointer `trace_name` di dalam ukuran bucket HDF5, SNR ∈ [-20, 80] dB | semua |
| R7 | `trace_has_offset = 1` dan `trace_missing_channel > 0` → **dipertahankan + di-flag** | comcat |
| R8 | Semua kolom asli dipertahankan (D10) + kolom turunan; kunci `(dataset, trace_name)` harus unik | semua |

Metadata sudah dicek ulang = versi terbaru (UW-ESS & Google Drive, SHA-256 identik, lihat `outputs/hdfs/lineage.jsonl`).

In [1]:
import sys, os, json
sys.path.insert(0, os.path.abspath('../src'))
from common import *
import pandas as pd, h5py
from pyspark.sql import functions as F, types as T
pd.set_option('display.max_rows', 100, 'display.width', 200)
spark = spark_session('05_clean')
raw = {d: spark.read.csv(f'{HDFS}{RAW}/{d}/{d}_metadata.csv', header=True, inferSchema=True) for d in DATASETS}
raw_n = {d: raw[d].count() for d in DATASETS}
raw_n

26/09/23 22:45:21 WARN Utils: Your hostname, Nakatas-MacBook-Pro.local resolves to a loopback address: 127.0.0.1; using 192.168.100.2 instead (on interface en0)
26/09/23 22:45:21 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/23 22:45:21 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


26/09/23 22:45:21 WARN SparkConf: Note that spark.local.dir will be overridden by the value set by the cluster manager (via SPARK_LOCAL_DIRS in mesos/standalone/kubernetes and LOCAL_DIRS in YARN).


{'comcat': 178939, 'exotic': 9267, 'noise': 51017}

## 1. Satukan skema

3 file punya kolom berbeda (37 / 17 / 11, bersarang). Kita pakai **skema ComCat (37 kolom) sebagai skema induk**: kolom yang tidak dimiliki Noise/Exotic diisi `NULL`, dan tipe disamakan (mis. `trace_P_arrival_sample` Exotic terbaca `double` `7000.000` → di-cast ke `int` seperti ComCat). Hasilnya 1 DataFrame yang bisa di-query bersama.

In [2]:
target = raw['comcat'].schema
def conform(df, d):
    cols = [(F.col(f'`{f.name}`').cast(f.dataType) if f.name in df.columns else F.lit(None).cast(f.dataType)).alias(f.name) for f in target]
    return df.select(F.lit(d).alias('dataset'), *cols)
allraw = conform(raw['comcat'], 'comcat').unionByName(conform(raw['exotic'], 'exotic')).unionByName(conform(raw['noise'], 'noise'))
# cast tidak boleh diam-diam mengubah nilai jadi NULL:
for d in ('exotic', 'noise'):
    for c in raw[d].columns:
        before = raw[d].filter(F.col(f'`{c}`').isNotNull()).count()
        after = allraw.filter((F.col('dataset') == d) & F.col(f'`{c}`').isNotNull()).count()
        assert before == after, (d, c, before, after)
print('Semua cast aman (tidak ada nilai yang hilang). Kolom:', len(allraw.columns))
allraw.groupBy('dataset').count().show()

Semua cast aman (tidak ada nilai yang hilang). Kolom: 38


+-------+------+
|dataset| count|
+-------+------+
| comcat|178939|
| exotic|  9267|
|  noise| 51017|
+-------+------+



## 2. Label ComCat (R1–R3)

### 2a. Dulu: crosstab baris yang `source_type`-nya tidak sama dengan arti kode PNSN
Tabel S2 (suplemen paper) = daftar resmi kode AQMS PNSN dan label ComCat-nya. Saat dikirim ke ComCat, beberapa tipe **digabung**:
- **earthquake** ← `eq` earthquake, `le` local, `re` regional, `ts` teleseism, `se` slow earthquake, `lp` long-period volcanic, `lf` low-frequency
- **explosion** ← `ex` explosion (generic chemical blast), `px` probable explosion, `sh` survey shot
- Kode lain punya label ComCat sendiri: `qb` quarry blast, `su` surface event, `sn` sonic boom, `th` thunder, `uk` unknown, …

In [3]:
S2 = {**{k: 'earthquake' for k in ['eq', 'le', 're', 'ts', 'se', 'lp', 'lf']},
      **{k: 'explosion' for k in ['ex', 'px', 'sh']},
      'su': 'surface event', 'th': 'thunder', 'sn': 'sonic boom', 'pc': 'plane crash', 'qb': 'quarry blast',
      'nt': 'nuclear test', 've': 'volcanic eruption', 'co': 'mine collapse', 'df': 'debris flow/avalanche', 'av': 'snow/ice avalanche',
      'ls': 'landslide', 'rb': 'rockburst', 'rs': 'rockslide', 'bc': 'building collapse', 'mi': 'meteor impact', 'uk': 'unknown'}
s2 = F.create_map([F.lit(x) for kv in S2.items() for x in kv])
cc = allraw.filter("dataset='comcat'").withColumn('pnsn_comcat_label', s2[F.col('source_type_pnsn_label')])
assert cc.filter('pnsn_comcat_label IS NULL').count() == 0, 'ada kode PNSN yang tidak ada di Tabel S2'

def label_case():
    # consistent = kode persis (eq/ex); mapping_only = beda kode tapi Tabel S2 menggabungkan ke kelas yang sama
    return (F.when(F.col('source_type_pnsn_label').isin('eq', 'ex') & (F.col('source_type') == F.col('pnsn_comcat_label')), 'consistent')
             .when(F.col('source_type') == F.col('pnsn_comcat_label'), 'mapping_only')
             .when(F.col('pnsn_comcat_label').isin('earthquake', 'explosion'), 'relabeled')
             .otherwise('dropped'))
cc = cc.withColumn('label_case', label_case())
not_same = cc.filter("label_case != 'consistent'")
print('Baris yang kodenya tidak persis sama dengan source_type:', not_same.count())
not_same.groupBy('source_type').pivot('source_type_pnsn_label').count().fillna(0).show()

Baris yang kodenya tidak persis sama dengan source_type: 16975


+-----------+---+---+-----+---+---+---+---+
|source_type| eq| lf|   px| qb| sn| su| uk|
+-----------+---+---+-----+---+---+---+---+
| earthquake|  0|758|  326|  0|  4| 72|  3|
|  explosion| 41|  0|15750| 21|  0|  0|  0|
+-----------+---+---+-----+---+---+---+---+



In [4]:
cases = (cc.groupBy('label_case', 'source_type', 'source_type_pnsn_label', 'pnsn_comcat_label').count()
           .orderBy('label_case', F.desc('count')).toPandas())
events = cc.groupBy('label_case').agg(F.countDistinct('event_id').alias('events')).toPandas()
cases

,label_case,source_type,source_type_pnsn_label,pnsn_comcat_label,count
0,consistent,earthquake,eq,earthquake,161901
1,consistent,explosion,ex,explosion,63
2,dropped,earthquake,su,surface event,72
3,dropped,explosion,qb,quarry blast,21
4,dropped,earthquake,sn,sonic boom,4
5,dropped,earthquake,uk,unknown,3
6,mapping_only,explosion,px,explosion,15750
7,mapping_only,earthquake,lf,earthquake,758
8,relabeled,earthquake,px,explosion,326
9,relabeled,explosion,eq,earthquake,41


**Cara membaca:**
- `mapping_only` → **bukan inkonsistensi**. Kodenya berbeda (`lf`, `px`) tapi menurut Tabel S2 memang digabung ke kelas yang sama. Tetap dipakai, label tidak berubah.
- `relabeled` → konflik sungguhan earthquake ↔ explosion. Label diambil dari PNSN (R2).
- `dropped` → PNSN bilang bukan earthquake/explosion (`su`, `sn`, `uk`, dan `qb` quarry blast, yang di ComCat punya label sendiri). Dibuang (R3).

⚠ Koreksi dari notebook 02: di sana `qb` sempat dianggap explosion (446 baris konflik). Menurut Tabel S2, `qb` adalah tipe tersendiri → konflik sebenarnya **467 baris**.

In [5]:
case_sum = cases.groupby('label_case')['count'].sum().rename('rows').to_frame().join(events.set_index('label_case')).reset_index()
case_sum

,label_case,rows,events
0,consistent,161964,39385
1,dropped,100,31
2,mapping_only,16508,6558
3,relabeled,367,80


## 3. Terapkan aturan & catat funnel

Setiap aturan dijalankan berurutan, dan jumlah baris dicatat setelah tiap langkah → **funnel**. Syaratnya: *baris awal − dibuang per aturan = baris akhir*, untuk setiap dataset.

In [6]:
funnel = []
def step(df, rule, desc):
    n = df.groupBy('dataset').count().toPandas().set_index('dataset')['count'].reindex(DATASETS, fill_value=0)
    funnel.append({'step': rule, 'description': desc, **n.to_dict()})
    return df

TO4 = F.create_map([F.lit(x) for kv in {'earthquake': 'earthquake', 'explosion': 'explosion',
                                         'surface event': 'surface_event', 'noise': 'noise'}.items() for x in kv])
df = step(allraw, 'R0', 'Raw (3 CSV digabung)')

# R1–R3 comcat: label = label ComCat versi PNSN; buang yang bukan earthquake/explosion
df = (df.withColumn('pnsn_comcat_label', s2[F.col('source_type_pnsn_label')])
        .withColumn('label_case', F.when(F.col('dataset') == 'comcat', label_case())))
df = step(df.filter("label_case IS NULL OR label_case != 'dropped'"), 'R3', 'ComCat: buang label PNSN di luar earthquake/explosion (su, sn, uk, qb)')
df = df.withColumn('label', F.when(F.col('dataset') == 'comcat', F.col('pnsn_comcat_label')).otherwise(F.col('source_type')))
df = df.withColumn('label', TO4[F.col('label')])

# R4 exotic: hanya surface event
df = step(df.filter('label IS NOT NULL'), 'R4', 'Exotic: buang sonic boom, thunder, plane crash')

# R5 noise: buang semua baris yang trace_name-nya muncul > 1 kali dalam dataset yang sama
dup = df.groupBy('dataset', 'trace_name').count().filter('count > 1').drop('count').withColumn('is_dup', F.lit(True))
df = df.join(dup, ['dataset', 'trace_name'], 'left')
df = step(df.filter('is_dup IS NULL').drop('is_dup'), 'R5', 'Buang semua baris dengan trace_name duplikat (noise: 395 pasangan)')
pd.DataFrame(funnel)

,step,description,comcat,exotic,noise
0,R0,Raw (3 CSV digabung),178939,9267,51017
1,R3,ComCat: buang label PNSN di luar earthquake/ex...,178839,9267,51017
2,R4,"Exotic: buang sonic boom, thunder, plane crash",178839,8912,51017
3,R5,Buang semua baris dengan trace_name duplikat (...,178839,8912,50227


## 4. Kolom turunan & validasi (R6, R7)

Kolom baru (kolom asli **tidak diubah**):
- `bucket`, `row`, `n_samples` ← dipecah dari `trace_name` (alamat waveform di HDF5)
- `snr_e`, `snr_n`, `snr_z` ← dipecah dari `trace_snr_db` (`"E|N|Z"`, `nan` → NULL)
- `flag_has_offset`, `flag_missing_channel` ← flag kualitas (ComCat saja; untuk Noise/Exotic dihitung dari waveform di notebook 04)
- `label`, `label_case`, `pnsn_comcat_label` ← asal-usul label

In [7]:
snr = F.split('trace_snr_db', r'\|')
df = (df.withColumn('bucket', F.regexp_extract('trace_name', r'^(bucket\d+)\$', 1))
        .withColumn('row', F.regexp_extract('trace_name', r'\$(\d+),', 1).cast('int'))
        .withColumn('n_samples', F.regexp_extract('trace_name', r':(\d+)$', 1).cast('int'))
        .select('*', *[F.when(snr[i] == 'nan', None).otherwise(snr[i].cast('double')).alias(f'snr_{c}') for i, c in enumerate('enz')])
        .withColumn('flag_has_offset', F.col('trace_has_offset') == 1)
        .withColumn('flag_missing_channel', F.col('trace_missing_channel') > 0))

# ukuran tiap bucket HDF5 (hanya baca metadata HDF5 lewat WebHDFS, bukan isinya)
fs = webhdfs(); bsize = []
for d in DATASETS:
    with fs.open(f'{RAW}/{d}/{d}_waveforms.hdf5', 'rb') as fo, h5py.File(fo, 'r') as f:
        bsize += [(d, b, f['data'][b].shape[0], f['data'][b].shape[2]) for b in f['data']]
bsize = spark.createDataFrame(bsize, 'dataset string, bucket string, bucket_rows int, bucket_samples int')
df = df.join(F.broadcast(bsize), ['dataset', 'bucket'], 'left')

EXPECTED_SAMPLES = F.when(F.col('dataset') == 'exotic', 18001).otherwise(15001)
valid = {
  'sampling_rate != 100 Hz':                 F.col('trace_sampling_rate_hz') != 100,
  'n_samples != expected (15001/18001)':     F.col('n_samples') != EXPECTED_SAMPLES,
  'n_samples != HDF5 array length':          F.col('n_samples') != F.col('bucket_samples'),
  'bucket not found in HDF5':                F.col('bucket_rows').isNull(),
  'row >= bucket size (pointer out of range)': F.col('row') >= F.col('bucket_rows'),
  'SNR above 80 dB':                F.greatest(*[F.col(f'snr_{c}') for c in 'enz']) > 80,
  'SNR below -20 dB':                        F.least(*[F.col(f'snr_{c}') for c in 'enz']) < -20,
  'trace_start_time is NULL':                F.col('trace_start_time').isNull(),
}
viol = pd.DataFrame([{'check': k, **df.groupBy('dataset').agg(F.sum(F.coalesce(v, F.lit(False)).cast('int')).alias('n'))
                      .toPandas().set_index('dataset')['n'].reindex(DATASETS, fill_value=0).to_dict()} for k, v in valid.items()])
viol

,check,comcat,exotic,noise
0,sampling_rate != 100 Hz,0,0,0
1,n_samples != expected (15001/18001),0,0,0
2,n_samples != HDF5 array length,0,0,0
3,bucket not found in HDF5,0,0,0
4,row >= bucket size (pointer out of range),0,0,0
5,SNR above 80 dB,0,0,0
6,SNR below -20 dB,0,0,0
7,trace_start_time is NULL,0,0,0


In [8]:
assert viol[DATASETS].to_numpy().sum() == 0, 'ada pelanggaran validitas — periksa sebelum lanjut'
df = step(df, 'R6', 'Validasi (sampling rate, jumlah sampel, pointer HDF5, SNR) — 0 pelanggaran, tidak ada yang dibuang')
flags = df.groupBy('dataset').agg(F.sum(F.col('flag_has_offset').cast('int')).alias('flag_has_offset'),
                                  F.sum(F.col('flag_missing_channel').cast('int')).alias('flag_missing_channel')).toPandas()
flags  # R7: dipertahankan, hanya ditandai

,dataset,flag_has_offset,flag_missing_channel
0,comcat,1534.0,53453.0
1,exotic,NaN,NaN
2,noise,NaN,NaN


## 5. Keunikan kunci & tulis ke processed zone

Kunci `(dataset, trace_name)` harus unik → setiap baris metadata menunjuk tepat 1 waveform, dan setiap waveform paling banyak 1 baris.
Ditulis sebagai **Parquet, dipartisi `dataset`** → folder `dataset=comcat/`, `dataset=exotic/`, `dataset=noise/`.

In [9]:
clean = df.drop('bucket_rows', 'bucket_samples')
assert clean.groupBy('dataset', 'trace_name').count().filter('count > 1').count() == 0
assert clean.groupBy('dataset', 'bucket', 'row').count().filter('count > 1').count() == 0
OUT = f'{HDFS}{PROCESSED}/metadata'
clean.repartition('dataset').write.mode('overwrite').partitionBy('dataset').parquet(OUT)
back = spark.read.parquet(OUT)
assert back.count() == clean.count()
print('ditulis:', back.count(), 'baris,', len(back.columns), 'kolom')
sh(f'hdfs dfs -ls -R {PROCESSED}/metadata | grep -v _SUCCESS')
sh(f'hdfs dfs -du -s -h {PROCESSED}/metadata')

ditulis: 237978 baris, 49 kolom


$ hdfs dfs -ls -R /datalake/processed/metadata | grep -v _SUCCESS
drwxr-xr-x   - axelnakata supergroup          0 2026-09-23 22:45 /datalake/processed/metadata/dataset=comcat
-rw-r--r--   1 axelnakata supergroup   12173065 2026-09-23 22:45 /datalake/processed/metadata/dataset=comcat/part-00001-61af3834-93ff-48e2-bf54-8530ba9a33e8.c000.snappy.parquet
drwxr-xr-x   - axelnakata supergroup          0 2026-09-23 22:45 /datalake/processed/metadata/dataset=exotic
-rw-r--r--   1 axelnakata supergroup     442940 2026-09-23 22:45 /datalake/processed/metadata/dataset=exotic/part-00000-61af3834-93ff-48e2-bf54-8530ba9a33e8.c000.snappy.parquet
drwxr-xr-x   - axelnakata supergroup          0 2026-09-23 22:45 /datalake/processed/metadata/dataset=noise
-rw-r--r--   1 axelnakata supergroup     854857 2026-09-23 22:45 /datalake/processed/metadata/dataset=noise/part-00000-61af3834-93ff-48e2-bf54-8530ba9a33e8.c000.snappy.parquet


$ hdfs dfs -du -s -h /datalake/processed/metadata
12.8 M  12.8 M  /datalake/processed/metadata


'12.8 M  12.8 M  /datalake/processed/metadata'

In [10]:
back.printSchema()

root
 |-- bucket: string (nullable = true)
 |-- trace_name: string (nullable = true)
 |-- event_id: string (nullable = true)
 |-- source_origin_time: timestamp (nullable = true)
 |-- source_latitude_deg: double (nullable = true)
 |-- source_longitude_deg: double (nullable = true)
 |-- source_type: string (nullable = true)
 |-- source_depth_km: double (nullable = true)
 |-- preferred_source_magnitude: double (nullable = true)
 |-- preferred_source_magnitude_type: string (nullable = true)
 |-- preferred_source_magnitude_uncertainty: double (nullable = true)
 |-- source_depth_uncertainty_km: double (nullable = true)
 |-- source_horizontal_uncertainty_km: double (nullable = true)
 |-- station_network_code: string (nullable = true)
 |-- station_channel_code: string (nullable = true)
 |-- station_code: string (nullable = true)
 |-- station_location_code: string (nullable = true)
 |-- station_latitude_deg: double (nullable = true)
 |-- station_longitude_deg: double (nullable = true)
 |-- stat

## 6. Funnel & laporan kualitas data (Bab 3.4)

In [11]:
fun = pd.DataFrame(funnel)
for d in DATASETS:
    fun[f'{d}_dropped'] = (-fun[d].diff()).fillna(0).astype(int)
fun['total'] = fun[DATASETS].sum(axis=1)
# Cek: awal − total dibuang = akhir, per dataset
for d in DATASETS:
    assert fun[d].iloc[0] - fun[f'{d}_dropped'].sum() == fun[d].iloc[-1] == back.filter(F.col('dataset') == d).count()
fun

,step,description,comcat,exotic,noise,comcat_dropped,exotic_dropped,noise_dropped,total
0,R0,Raw (3 CSV digabung),178939,9267,51017,0,0,0,239223
1,R3,ComCat: buang label PNSN di luar earthquake/ex...,178839,9267,51017,100,0,0,239123
2,R4,"Exotic: buang sonic boom, thunder, plane crash",178839,8912,51017,0,355,0,238768
3,R5,Buang semua baris dengan trace_name duplikat (...,178839,8912,50227,0,0,790,237978
4,R6,"Validasi (sampling rate, jumlah sampel, pointe...",178839,8912,50227,0,0,0,237978


In [12]:
def dist(x):
    return x.groupBy('dataset', 'label').count().toPandas()
before_lab = allraw.withColumn('label', F.col('source_type')).groupBy('dataset', 'label').count().toPandas()
after_lab = dist(back).sort_values(['dataset', 'count'], ascending=[True, False])
after_lab['pct_of_total'] = (after_lab['count'] / after_lab['count'].sum() * 100).round(2)
after_lab

,dataset,label,count,pct_of_total
0,comcat,earthquake,162700,68.37
1,comcat,explosion,16139,6.78
3,exotic,surface_event,8912,3.74
2,noise,noise,50227,21.11


In [13]:
# Orphan: slot waveform di HDF5 yang tidak dirujuk metadata (sebelum vs sesudah cleaning)
slots = bsize.groupBy('dataset').agg(F.sum('bucket_rows').alias('hdf5_traces')).toPandas().set_index('dataset')
ref_before = allraw.withColumn('slot', F.regexp_extract('trace_name', r'^([^,]+),', 1)).groupBy('dataset').agg(F.countDistinct('slot').alias('ref')).toPandas().set_index('dataset')
ref_after = back.groupBy('dataset').agg(F.countDistinct('bucket', 'row').alias('ref')).toPandas().set_index('dataset')
orph = pd.DataFrame({'hdf5_traces': slots.hdf5_traces, 'orphans_before': slots.hdf5_traces - ref_before.ref,
                     'orphans_after': slots.hdf5_traces - ref_after.ref}).reindex(DATASETS)
orph['readme_removed_nov2025'] = [4970, 0, 445]
orph['unexplained'] = orph.orphans_before - orph.readme_removed_nov2025
orph

,hdf5_traces,orphans_before,orphans_after,readme_removed_nov2025,unexplained
dataset,,,,,
comcat,187053,8114,8214,4970,3144
exotic,9517,250,605,0,250
noise,51462,840,1235,445,395


Orphan ComCat sebelum cleaning = 8,114, sedangkan README hanya menjelaskan 4,970 → **3,144 slot tidak terjelaskan** (tersebar merata di 10 bucket, isinya waveform nyata, bukan nol). Tidak berbahaya: konversi (notebook 04) hanya membaca slot yang dirujuk metadata bersih.

In [14]:
KEYCOLS = ['event_id', 'source_type', 'station_code', 'trace_name', 'trace_start_time', 'trace_P_arrival_sample', 'trace_snr_db', 'label']
def completeness(x, ds):
    x = x.filter(F.col('dataset') == ds); n = x.count()
    return {c: round(x.filter(F.col(c).isNull()).count() * 100 / n, 2) for c in KEYCOLS if c in x.columns}
comp = pd.DataFrame([{'dataset': d, 'stage': s, **completeness(t, d)} for d in DATASETS
                     for s, t in (('before', allraw.withColumn('label', F.col('source_type'))), ('after', back))])
comp  # null % kolom kunci; NULL di noise/exotic = kolom memang tidak ada di file aslinya

,dataset,stage,event_id,source_type,station_code,trace_name,trace_start_time,trace_P_arrival_sample,trace_snr_db,label
0,comcat,before,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0
1,comcat,after,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0
2,exotic,before,0.0,0.0,0.0,0.0,0.0,1.29,0.0,0.0
3,exotic,after,0.0,0.0,0.0,0.0,0.0,1.21,0.0,0.0
4,noise,before,100.0,0.0,0.0,0.0,0.0,100.00,100.0,0.0
5,noise,after,100.0,0.0,0.0,0.0,0.0,100.00,100.0,0.0


In [15]:
quality = pd.DataFrame([
 ('Completeness', 'null % kolom kunci', 'lihat tabel completeness', 'tidak berubah untuk kolom asli; label 0% null'),
 ('Validity', 'sampling rate, jumlah sampel, pointer HDF5, SNR range', f'{int(viol[DATASETS].to_numpy().sum())} pelanggaran', '0 pelanggaran'),
 ('Uniqueness', 'duplikat (dataset, trace_name)', f"noise {fun.loc[fun.step=='R5', 'noise_dropped'].item()} baris (395 nilai)", '0'),
 ('Consistency', 'label ComCat vs PNSN (Tabel S2)',
  ', '.join(f'{r.label_case} {r.rows:,}' for r in case_sum.itertuples()),
  f"{case_sum.set_index('label_case').loc['relabeled', 'rows']:,} di-relabel, {case_sum.set_index('label_case').loc['dropped', 'rows']:,} dibuang"),
 ('Orphans', 'slot HDF5 tanpa metadata', ', '.join(f'{d} {orph.loc[d, "orphans_before"]:,}' for d in DATASETS),
  ', '.join(f'{d} {orph.loc[d, "orphans_after"]:,}' for d in DATASETS)),
], columns=['dimension', 'metric', 'before', 'after'])
quality

,dimension,metric,before,after
0,Completeness,null % kolom kunci,lihat tabel completeness,tidak berubah untuk kolom asli; label 0% null
1,Validity,"sampling rate, jumlah sampel, pointer HDF5, SN...",0 pelanggaran,0 pelanggaran
2,Uniqueness,"duplikat (dataset, trace_name)",noise 790 baris (395 nilai),0
3,Consistency,label ComCat vs PNSN (Tabel S2),"consistent 161,964, dropped 100, mapping_only ...","367 di-relabel, 100 dibuang"
4,Orphans,slot HDF5 tanpa metadata,"comcat 8,114, exotic 250, noise 840","comcat 8,214, exotic 605, noise 1,235"


### Simpan laporan
- `outputs/tables/quality_report.md` + `cleaning_funnel.md` + `label_cases.md` (untuk laporan)
- `/datalake/quality/metadata_quality.json` (di HDFS, ikut data lake)

In [16]:
save_md(fun[['step', 'description', *DATASETS, 'total']], 'cleaning_funnel', 'Funnel cleaning metadata (jumlah baris setelah tiap aturan)',
        'Setiap angka bisa ditelusuri: baris awal − dibuang per aturan = baris akhir (diverifikasi dengan assert di notebook 03).')
save_md(cases, 'label_cases', 'Label ComCat vs kode PNSN (Tabel S2 Ni dkk. 2023)', case_sum.to_markdown(index=False))
body = ['# Laporan kualitas data metadata (sebelum vs sesudah cleaning)', '', quality.to_markdown(index=False), '',
        '## Validitas', '', viol.to_markdown(index=False), '', '## Completeness (null % kolom kunci)', '', comp.to_markdown(index=False), '',
        '## Orphan waveform', '', orph.reset_index(names='dataset').to_markdown(index=False), '',
        '## Flag kualitas (dipertahankan)', '', flags.to_markdown(index=False), '',
        '## Distribusi kelas setelah cleaning', '', after_lab.to_markdown(index=False), '']
open(f'{OUT_TABLES}/quality_report.md', 'w').write('\n'.join(body))

report = {'generated_by': 'notebooks/03_cleaning_quality.ipynb', 'raw_rows': raw_n,
          'clean_rows': after_lab.groupby('dataset')['count'].sum().to_dict(),
          'funnel': fun.to_dict('records'), 'label_cases': case_sum.to_dict('records'),
          'validity': viol.to_dict('records'), 'orphans': orph.reset_index(names='dataset').to_dict('records'),
          'flags': flags.to_dict('records'), 'class_distribution': after_lab.to_dict('records')}
local = f'{OUT_TABLES}/metadata_quality.json'
json.dump(report, open(local, 'w'), indent=1, default=int)
sh(f'hdfs dfs -put -f {local} {QUALITY}/metadata_quality.json && hdfs dfs -ls {QUALITY}')

$ hdfs dfs -put -f /Users/axelnakata/Documents/BigData-Earthquake/outputs/tables/metadata_quality.json /datalake/quality/metadata_quality.json && hdfs dfs -ls /datalake/quality
Found 1 items
-rw-r--r--   1 axelnakata supergroup       3873 2026-09-23 22:45 /datalake/quality/metadata_quality.json


'Found 1 items\n-rw-r--r--   1 axelnakata supergroup       3873 2026-09-23 22:45 /datalake/quality/metadata_quality.json'

## Ringkasan

- Metadata bersih ada di `/datalake/processed/metadata/dataset={comcat,exotic,noise}/` (Parquet, semua kolom asli + kolom turunan).
- Setiap baris yang dibuang bisa dijelaskan oleh 1 aturan (funnel), dan raw zone tidak disentuh.
- Label ComCat mengikuti Tabel S2: `mapping_only` tidak dihitung konflik, konflik earthquake↔explosion mengikuti PNSN, selain itu dibuang.
- Langkah berikutnya (notebook 04): konversi waveform HDF5 → Parquet, **dikendalikan oleh tabel ini** (hanya slot yang dirujuk).

In [17]:
spark.stop()